# ESRGAN Single-Image Enhancement Pipeline
This notebook is streamlined for one image at a time and works in Colab or Kaggle.

Pipeline:
1. Clone BasicSR
2. Install dependencies
3. Download ESRGAN pretrained model
4. Provide one image (upload or direct path)
5. Run enhancement with GPU and CPU fallback
6. Save and preview input vs output

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if Path('/content').exists():
    base_dir = Path('/content')
elif Path('/kaggle/working').exists():
    base_dir = Path('/kaggle/working')
else:
    base_dir = Path.cwd()

repo_dir = base_dir / 'BasicSR'
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(['git', 'clone', 'https://github.com/XPixelGroup/BasicSR.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import subprocess
import sys
import torch

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, 'setup.py', 'develop'], check=True)

print('Torch Version:', torch.__version__)
print('CUDA Version:', torch.version.cuda)
print('CUDNN Version:', torch.backends.cudnn.version())
print('CUDA Available:', torch.cuda.is_available())

In [ ]:
import subprocess
import sys
import urllib.request
from pathlib import Path

# 1) ESRGAN weights: use BasicSR official downloader (Google Drive IDs maintained by repo).
subprocess.run([sys.executable, 'scripts/download_pretrained_models.py', 'ESRGAN'], check=True)

# 2) RealESRGAN x4plus weights with fallback release tags.
real_esrgan_x4plus_path = Path('experiments/pretrained_models/RealESRGAN/RealESRGAN_x4plus.pth')
real_esrgan_x4plus_path.parent.mkdir(parents=True, exist_ok=True)

x4plus_candidate_urls = [
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.3.0/RealESRGAN_x4plus.pth',
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus.pth',
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
]

if real_esrgan_x4plus_path.exists():
    print(f'RealESRGAN_x4plus: already exists -> {real_esrgan_x4plus_path}')
else:
    download_error = None
    for url in x4plus_candidate_urls:
        try:
            print(f'Trying RealESRGAN_x4plus from: {url}')
            urllib.request.urlretrieve(url, str(real_esrgan_x4plus_path))
            print(f'RealESRGAN_x4plus: downloaded -> {real_esrgan_x4plus_path}')
            download_error = None
            break
        except Exception as e:
            download_error = e
            print(f'Failed: {e}')

    if not real_esrgan_x4plus_path.exists():
        raise RuntimeError(
            'Could not download RealESRGAN_x4plus.pth from known URLs. '
            'Please verify network access and release links. Last error: '
            f'{download_error}'
        )

# 3) RealESRGAN general x4v3 (requested model).
real_esrgan_general_x4v3_path = Path('experiments/pretrained_models/RealESRGAN/realesr-general-x4v3.pth')
real_esrgan_general_x4v3_url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth'

if real_esrgan_general_x4v3_path.exists():
    print(f'RealESRGAN_general_x4v3: already exists -> {real_esrgan_general_x4v3_path}')
else:
    print(f'Downloading RealESRGAN_general_x4v3 from: {real_esrgan_general_x4v3_url}')
    urllib.request.urlretrieve(real_esrgan_general_x4v3_url, str(real_esrgan_general_x4v3_path))
    print(f'RealESRGAN_general_x4v3: downloaded -> {real_esrgan_general_x4v3_path}')

## Input Mode
- Set `USE_UPLOAD = True` to upload one image from Colab file picker.
- Set `USE_UPLOAD = False` to use a direct image path (recommended for Kaggle).

In [ ]:
USE_UPLOAD = True
INPUT_IMAGE_PATH = ''

OUTPUT_DIR = 'results/compare'

# Choose one or more models from MODEL_CONFIGS keys.
SELECTED_MODELS = ['ESRGAN', 'RealESRGAN_x4plus', 'RealESRGAN_general_x4v3']

MODEL_CONFIGS = {
    'ESRGAN': {
        'arch': 'rrdb',
        'model_path': 'experiments/pretrained_models/ESRGAN/ESRGAN_SRx4_DF2KOST_official-ff704c30.pth',
        'num_block': 23,
        'num_grow_ch': 32,
        'state_key_priority': ['params', 'params_ema']
    },
    'RealESRGAN_x4plus': {
        'arch': 'rrdb',
        'model_path': 'experiments/pretrained_models/RealESRGAN/RealESRGAN_x4plus.pth',
        'num_block': 23,
        'num_grow_ch': 32,
        'state_key_priority': ['params_ema', 'params']
    },
    'RealESRGAN_general_x4v3': {
        'arch': 'srvgg',
        'model_path': 'experiments/pretrained_models/RealESRGAN/realesr-general-x4v3.pth',
        'num_conv': 32,
        'upscale': 4,
        'act_type': 'prelu',
        'state_key_priority': ['params_ema', 'params']
    }
}

In [ ]:
from pathlib import Path

upload_dir = Path('datasets/upload')
upload_dir.mkdir(parents=True, exist_ok=True)

if USE_UPLOAD:
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No file uploaded.')

        first_name = next(iter(uploaded))
        dst = upload_dir / first_name
        with open(dst, 'wb') as f:
            f.write(uploaded[first_name])
        INPUT_IMAGE_PATH = str(dst)
        print(f'Uploaded image: {INPUT_IMAGE_PATH}')
    except Exception as e:
        print('Upload helper works in Colab. For Kaggle/Jupyter, set INPUT_IMAGE_PATH and rerun this cell.')
        print(f'Upload skipped: {e}')
else:
    if not INPUT_IMAGE_PATH:
        raise ValueError('Set INPUT_IMAGE_PATH when USE_UPLOAD is False.')
    print(f'Using image path: {INPUT_IMAGE_PATH}')

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import torch
from basicsr.archs.rrdbnet_arch import RRDBNet
from basicsr.archs.srvgg_arch import SRVGGNetCompact

if not INPUT_IMAGE_PATH:
    raise ValueError('No input image selected. Run the input cell and provide/upload one image.')

if not SELECTED_MODELS:
    raise ValueError('SELECTED_MODELS is empty. Pick at least one model.')

invalid_models = [m for m in SELECTED_MODELS if m not in MODEL_CONFIGS]
if invalid_models:
    raise ValueError(f'Unknown model(s): {invalid_models}. Available: {list(MODEL_CONFIGS.keys())}')

input_path = Path(INPUT_IMAGE_PATH)
if not input_path.exists():
    raise FileNotFoundError(f'Input image not found: {input_path}')

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')

img = cv2.imread(str(input_path), cv2.IMREAD_COLOR)
if img is None:
    raise ValueError(f'Failed to read image: {input_path}')

img_tensor = img.astype(np.float32) / 255.0
img_tensor = torch.from_numpy(np.transpose(img_tensor[:, :, [2, 1, 0]], (2, 0, 1))).float()
img_tensor = img_tensor.unsqueeze(0).to(device)

def load_state_dict_for_model(model_path, state_key_priority):
    state = torch.load(model_path, map_location=device)
    if isinstance(state, dict):
        for key in state_key_priority:
            if key in state:
                return state[key]
    return state

def build_model(cfg):
    if cfg['arch'] == 'rrdb':
        return RRDBNet(
            num_in_ch=3,
            num_out_ch=3,
            num_feat=64,
            num_block=cfg['num_block'],
            num_grow_ch=cfg['num_grow_ch']
        )
    if cfg['arch'] == 'srvgg':
        return SRVGGNetCompact(
            num_in_ch=3,
            num_out_ch=3,
            num_feat=64,
            num_conv=cfg['num_conv'],
            upscale=cfg['upscale'],
            act_type=cfg['act_type']
        )
    raise ValueError(f"Unsupported arch: {cfg['arch']}")

OUTPUT_PATHS = {}
LAST_INPUT_PATH = str(input_path)

for model_name in SELECTED_MODELS:
    cfg = MODEL_CONFIGS[model_name]
    model_path = cfg['model_path']
    if not Path(model_path).exists():
        raise FileNotFoundError(f'Model weights not found: {model_path}')

    model = build_model(cfg)
    state_dict = load_state_dict_for_model(model_path, cfg['state_key_priority'])
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    model = model.to(device)

    with torch.no_grad():
        output = model(img_tensor)

    output = output.data.squeeze().float().cpu().clamp_(0, 1).numpy()
    output = np.transpose(output[[2, 1, 0], :, :], (1, 2, 0))
    output = (output * 255.0).round().astype(np.uint8)

    output_path = output_dir / f'{input_path.stem}_{model_name}.png'
    cv2.imwrite(str(output_path), output)
    OUTPUT_PATHS[model_name] = str(output_path)
    print(f'{model_name}: saved -> {output_path}')

In [ ]:
import cv2
import matplotlib.pyplot as plt

def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'Cannot read image: {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

if 'OUTPUT_PATHS' not in globals() or not OUTPUT_PATHS:
    raise ValueError('No outputs found. Run the inference cell first.')

img_input = read_rgb(LAST_INPUT_PATH)
model_names = list(OUTPUT_PATHS.keys())

n_cols = 1 + len(model_names)
fig = plt.figure(figsize=(6 * n_cols, 6))

ax = fig.add_subplot(1, n_cols, 1)
ax.set_title('Input image')
ax.axis('off')
ax.imshow(img_input)

for i, model_name in enumerate(model_names, start=2):
    ax = fig.add_subplot(1, n_cols, i)
    ax.set_title(model_name)
    ax.axis('off')
    ax.imshow(read_rgb(OUTPUT_PATHS[model_name]))

plt.tight_layout()
plt.show()

print('Saved outputs:')
for model_name in model_names:
    print(f'- {model_name}: {OUTPUT_PATHS[model_name]}')